Выбор большой языковой модели

In [1]:
from llamba.chatmodels.ollama import OllamaModel
chatbot = OllamaModel(url="http://127.0.0.1:11434/", 
                      endpoint="api/generate", 
                      model="deepseek-r1", 
                      num_threads=1, 
                      check_connection_timeout=15,  
                      request_timeout=60,
                      think=False)
connection = chatbot.check_connection()
print(connection)

True


Подготовка данных

In [2]:
import pandas as pd

mock_descriptions = {
    'CXCL9': {'description': 'gene for cancer', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CCL22': {'description': 'gene for ccl', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'IL6': {'description': 'gene for fish', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'PDGFB': {'description': 'gene for idkB', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CD40LG': {'description': 'gene for TV', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'IL27': {'description': 'gene for fishy fish', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'VEGFA': {'description': 'gene for monster', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CSF1': {'description': 'gene for racing', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'PDGFA': {'description': 'gene for idkA', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CXCL10': {'description': 'gene for 67', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'}}

desc_df = pd.DataFrame(mock_descriptions).T

In [3]:
desc_df

,description,low_shap_desc,high_shap_desc
CXCL9,gene for cancer,badbad,goodgood
CCL22,gene for ccl,badbad,goodgood
IL6,gene for fish,badbad,goodgood
PDGFB,gene for idkB,badbad,goodgood
CD40LG,gene for TV,badbad,goodgood
IL27,gene for fishy fish,badbad,goodgood
VEGFA,gene for monster,badbad,goodgood
CSF1,gene for racing,badbad,goodgood
PDGFA,gene for idkA,badbad,goodgood
CXCL10,gene for 67,badbad,goodgood


In [9]:
#import torch
#from torch import nn
import numpy as np

#from llamba_library.bioage_model import BioAgeModel
#from llamba_library.functions import get_shap_dict
#from txai_omics_3.models.tabular.widedeep.ft_transformer \
#    import WDFTTransformerModel, FN_SHAP, FN_CHECKPOINT, TRAIN_DATA_PATH


#### Данные
feats = [
    'CXCL9',
    'CCL22',
    'IL6',
    'PDGFB',
    'CD40LG',
    'IL27',
    'VEGFA',
    'CSF1',
    'PDGFA',
    'CXCL10'
]

#dataset_path = TRAIN_DATA_PATH
#train_data = pd.read_excel(TRAIN_DATA_PATH)
#train_data = train_data[train_data['Split'] == 'trn_val'].copy()
#train_data = train_data.loc[:, feats + ['Age']]

my_data = {'CXCL9': 2599.629474, 
           'CCL22': 820.306524, 
           'IL6': 0.846377, 
           'PDGFB': 13400.666359, 
           'CD40LG': 1853.847406, 
           'IL27': 1128.886982,
           'VEGFA': 153.574220,
           'CSF1': 239.627236,
           'PDGFA': 1005.844290,
           'CXCL10': 228.229829,
           'Age': 90.454972 }


data = pd.DataFrame(my_data, index=[0])

#### Модель
    
#fn_model = FN_CHECKPOINT
#model = WDFTTransformerModel.load_from_checkpoint(fn_model)
#bioage_model = BioAgeModel(model=model)

def predict_func(x):
    batch = {
        'all': torch.from_numpy(np.float32(x)),
        'continuous': torch.from_numpy(np.float32(x))
    }
    return model(batch).cpu().detach().numpy()

top_n = 3 # количество признаков с наибольшим вкладом

In [14]:
data.T

,0
CXCL9,2599.629474
CCL22,820.306524
IL6,0.846377
PDGFB,13400.666359
CD40LG,1853.847406
IL27,1128.886982
VEGFA,153.574220
CSF1,239.627236
PDGFA,1005.844290
CXCL10,228.229829


In [15]:
desc_df

,description,low_shap_desc,high_shap_desc
CXCL9,gene for cancer,badbad,goodgood
CCL22,gene for ccl,badbad,goodgood
IL6,gene for fish,badbad,goodgood
PDGFB,gene for idkB,badbad,goodgood
CD40LG,gene for TV,badbad,goodgood
IL27,gene for fishy fish,badbad,goodgood
VEGFA,gene for monster,badbad,goodgood
CSF1,gene for racing,badbad,goodgood
PDGFA,gene for idkA,badbad,goodgood
CXCL10,gene for 67,badbad,goodgood


Инициализация коннектора

In [3]:
from llamba.connector import LlambaConnector

connector = LlambaConnector(language="en", bioage_model=bioage_model, chat_model=chatbot)

Инициализация векторной базы данных

In [4]:
from llamba.util.document_repo import DocumentRepoQdrant

document_repo = DocumentRepoQdrant(url="127.0.0.1", 
                                   port="6060", 
                                   collection_name="ageing_docs",
                                   payload_return="data",
                                   max_results=5)
connector.specify_db(document_repo)

Указание калькулятора биологических часов

In [5]:
connector.specify_clock_name(name="Immunoage", doi="10.1007/s11357-022-00540-4")

Указание производственных условий

In [6]:
connector.specify_conditions(condition="mining industry", duration="15 years")

Передача экспертной системе данных 

In [7]:
res = connector.analyze(data=data, top_n=3, analyze_feats=True, analyze_risks=True, train_data=train_data, predict_func=predict_func)
print(res['analysis'])

1) Bioage: 79 years.  
Age Acceleration: -11 years (indicating slower aging than normal).

2) Most influential parameters:  

| Feature       | Value          | Age Acceleration |
|---------------|----------------|------------------|
| IL6           | 0.846377       | -5.09498         |
| CD40LG        | 1853.847406    | 7.23675          |
| CXCL9         | 2599.629474    | -9.89109         |

3) Description of parameters:  

- **IL6**: A pro-inflammatory cytokine. Higher levels may indicate chronic inflammation, increasing disease risk. Lower levels (normal aging) reduce risk. Acceleration below normal (e.g., -5.09498) suggests reduced inflammation, lowering disease risk.  
- **CD40LG**: A costimulatory molecule for immune responses. Elevated levels may indicate immune dysregulation, increasing autoimmune or malignancy risk. Acceleration above normal (e.g., 7.23675) suggests immune acceleration, potentially increasing susceptibility to immune-related disorders.  
- **CXCL9**: A chemok

In [8]:
recommendations = connector.produce_recommendations()
print(recommendations)

Based on the provided data, your immunological aging profile shows mixed signals with decelerated aging in some markers (IL-6, CXCL-9) and accelerated aging in others (CD40LG). This combination may influence disease risk, particularly for infections and immune-related conditions. Given the mining background, occupational exposures (e.g., silica, heavy metals) could contribute to immune dysregulation or chronic inflammation. Recommendations to normalize results and reduce disease risk:

1. **Immunomodulation**: Address immune dysregulation by optimizing T-cell and cytokine balance. The accelerated CD40LG suggests heightened immune activation, which could increase inflammation or autoimmunity. Consider anti-inflammatory therapies (e.g., low-dose corticosteroids or IL-6 inhibitors) to counteract potential pro-inflammatory effects, though IL-6's decelerated aging indicates reduced chronic inflammation. Monitor immune responses for infections (e.g., STDs, HIV) given conflicting signals.

2.